# Reproducibility and FitSpec

A figure in a manuscript shows a fitted dose-response curve. The methods section says "fitted with 4PL, 1/Y² weighting." Without knowing the exact software version, initial guess strategy, and random seed, no one can reproduce the exact curve. `openfit` solves this with `FitSpec` — a JSON reproducibility manifest that travels with every fit.

**What FitSpec contains:**

| Field | Description |
|---|---|
| `model_id` | The model name |
| `param_values` | Exact fitted parameter values (lossless `repr()` serialisation) |
| `weight_scheme` | The weight scheme string |
| `data_hash` | SHA-256 of the input arrays (float64 LE bytes) |
| `openfit_version` | openfit version pin |
| `scipy_version` | SciPy version pin |
| `numpy_version` | NumPy version pin |
| `random_seed` | Seed used for bootstrap CI |


In [ ]:
import numpy as np
import json
import hashlib
import struct
from openfit import Fit, FitSpec


## Step 1: Run a fit and examine the FitSpec


In [ ]:
# Simulate data (would normally be loaded from a file)
rng = np.random.default_rng(42)
x = np.array([0.1, 0.3, 1.0, 3.0, 10.0, 30.0, 100.0, 300.0])
y_true = 2.0 + 96.0 / (1 + (10.0 / x) ** 1.5)
y = y_true * (1 + rng.normal(0, 0.10, size=len(x)))
y = np.clip(y, 0.1, None)

# Run the fit
result = Fit("hill4p", x, y, weights="1/y2").run()
print(result.summary())


In [ ]:
# Inspect the FitSpec
spec = result.spec
print("=== FitSpec contents ===")
print(f"model_id:        {spec.model_id}")
print(f"weight_scheme:   {spec.weight_scheme}")
print(f"data_hash:       {spec.data_hash}")
print(f"openfit_version: {spec.openfit_version}")
print(f"scipy_version:   {spec.scipy_version}")
print(f"numpy_version:   {spec.numpy_version}")
print(f"random_seed:     {spec.random_seed}")
print(f"\nparam_values:")
for k, v in spec.param_values.items():
    print(f"  {k}: {v}")


## Step 2: Save the FitSpec to JSON

This is what you would deposit alongside a manuscript.


In [ ]:
spec_path = "/tmp/my_fit.spec.json"
spec.to_json(spec_path)

# Show the raw JSON
raw = open(spec_path).read()
print(raw)


## Step 3: Verify the data hash independently

Anyone with the data file can recompute the hash and confirm it matches the spec — proving the spec applies to their copy of the data.


In [ ]:
def compute_data_hash(x_arr, y_arr):
    """Recompute the openfit data hash for verification."""
    # openfit hashes x and y arrays as float64 little-endian bytes, concatenated
    x64 = x_arr.astype(np.float64)
    y64 = y_arr.astype(np.float64)
    raw_bytes = x64.tobytes() + y64.tobytes()
    return hashlib.sha256(raw_bytes).hexdigest()

computed_hash = compute_data_hash(x, y)
stored_hash = spec.data_hash

print(f"Computed hash: {computed_hash}")
print(f"Stored hash:   {stored_hash}")
print(f"Match: {computed_hash == stored_hash}")


## Step 4: Load the FitSpec and verify parameters

This simulates what a reader (or automated check) would do.


In [ ]:
# Load the spec from JSON (simulating a reader who has the file)
loaded_spec = FitSpec.from_json(open(spec_path).read())

print("Loaded spec model_id:", loaded_spec.model_id)
print("Loaded spec param_values:", loaded_spec.param_values)

# Compare with original fit
print("\nParameter agreement:")
for name, val in result.params.items():
    loaded_val = loaded_spec.param_values[name]
    match = abs(float(val) - float(loaded_val)) < 1e-10
    print(f"  {name}: fit={val:.6f}  spec={float(loaded_val):.6f}  match={match}")


## Step 5: Full reproducibility — rerun from spec

Show that running the same fit with the same inputs produces identical results (floating-point identical).


In [ ]:
# Rerun the fit from scratch with identical inputs
result2 = Fit(
    loaded_spec.model_id,
    x, y,
    weights=loaded_spec.weight_scheme,
    random_seed=loaded_spec.random_seed,
).run()

print("Original vs rerun parameters:")
print(f"{'Parameter':12s}  {'Original':>14s}  {'Rerun':>14s}  {'Identical':>10s}")
print("-" * 56)
for name in result.params:
    v1 = result.params[name]
    v2 = result2.params[name]
    identical = abs(v1 - v2) < 1e-12
    print(f"{name:12s}  {v1:>14.8f}  {v2:>14.8f}  {str(identical):>10s}")

print(f"\nData hashes match: {result.spec.data_hash == result2.spec.data_hash}")


## Practical workflow for a manuscript

Here is the recommended end-to-end workflow for making a published fit fully reproducible:

1. Run `result = Fit(...).run()`
2. Save `result.spec.to_json("figure2_fit.spec.json")`
3. Deposit the raw data CSV **and** the spec JSON on Zenodo/OSF as supplementary material
4. In the methods section, note:
   > "Fitted with openfit v0.1.2 (doi:10.5281/zenodo.XXXXXXX); FitSpec deposited at [URL]"
5. Any reader can verify: load the spec, recompute the data hash, confirm parameters.

The spec JSON is small (< 1 KB) and human-readable, making it a lightweight but complete reproducibility receipt.


In [ ]:
# What to include in supplementary materials:
print("Files to deposit alongside the manuscript:")
print("  data/dose_response_raw.csv   — raw x, y values")
print("  data/figure2_fit.spec.json   — FitSpec reproducibility manifest")
print()
print("FitSpec JSON snippet for methods section:")
spec_dict = json.loads(open(spec_path).read())
snippet = {k: spec_dict[k] for k in ["model_id", "weight_scheme", "data_hash",
                                       "openfit_version", "scipy_version"]}
print(json.dumps(snippet, indent=2))


## Summary

- `FitSpec` makes every openfit result auditable and reproducible.
- The **data hash** lets readers verify they have the exact same dataset.
- **Version pins** document the software environment for long-term reproducibility.
- Deposit `.spec.json` alongside your data on Zenodo, Figshare, or OSF.
- This is what "every result you can cite" means in practice.
